In [1]:
#Dynamic search path configuration
import sys
import os

#Adjusting search path based on notebook's depth in directory --> iterates through directories until it finds directory name: US_conflict_forecasting
current_dir = os.getcwd()
while 'US_conflict_forecasting' not in os.path.basename(current_dir):
    current_dir = os.path.dirname(current_dir)

#Adjusting search path based on 'new' current directory
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from project_setup import setup_project
project_root_dir = setup_project()

Current Project Root Directory set to: /Users/luca/Documents/GitHub/US_conflict_forecasting
Current Search Path set to: ['/Users/luca/Documents/GitHub/US_conflict_forecasting', '/Users/luca/Documents/GitHub/US_conflict_forecasting/1_Features/b_Conflict_Features/Conflict_PreviousYear', '/Users/luca/anaconda3/envs/ML_Lab/lib/python312.zip', '/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12', '/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12/lib-dynload', '', '/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12/site-packages', '/Users/luca/Documents/GitHub/US_conflict_forecasting']


In [2]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import ast
import folium

In [3]:
#Load Data
ACLED_20_23_all = pd.read_csv(os.path.join(project_root_dir, '2_Dataset/a_Input_Datasets/US2020_2023.csv'))

#Clean Data, filtering for PA and 2023!
ACLED_21_PA = ACLED_20_23_all[(ACLED_20_23_all['admin1'] == 'Pennsylvania') & (ACLED_20_23_all['year'] == 2021)]

#Transform ACLED Data into GeoDataframe
#iterating over each row of the ACLED dataset to create list of point objects using lat and lon values
geometry = [Point(xy) for xy in zip(ACLED_21_PA['longitude'], ACLED_21_PA['latitude'])]

#creating new Geo dataframe using the Geodatapoints extracted in the previous step
gdf_ACLED_21_PA = gpd.GeoDataFrame(ACLED_21_PA, geometry=geometry)

### Parsing Vector Layer (ZIP Code Polygons)

In [4]:
#Parsing Geojson file 
gdf_ZIP_PA = gpd.read_file(os.path.join(project_root_dir, '2_Dataset/a_Input_Datasets/pa_pennsylvania_zip_codes_geo.min.json'))

### Overlaying Maps

In [5]:
#Defining center of map using bounds of the ZIP Code polygons --> calculating by averaging lat and long values to find midpoint
bounds = gdf_ZIP_PA.total_bounds
center = [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]

#Initialising map centrered around the calculated center 
map_PA = folium.Map(location=center, zoom_start=10)

#Adding ZIP Code polygon layer to initialised map 
folium.GeoJson(gdf_ZIP_PA, name="ZIP Codes PA").add_to(map_PA)

#Adding ACLED data points to initialised map 
for idx, row in ACLED_21_PA.iterrows():
    folium.Marker([row['latitude'], row['longitude']]).add_to(map_PA)

### Spatial Join 

In [6]:
spatial_join_PA_21 = gpd.sjoin(gdf_ACLED_21_PA, gdf_ZIP_PA, how="inner", op='within')

/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3493: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if await self.run_code(code, result, async_=asy):
/var/folders/4s/d5gptfss61l9fqz7rk3r5m880000gn/T/ipykernel_33695/1419187888.py:1: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: None
Right CRS: EPSG:4326

  spatial_join_PA_21 = gpd.sjoin(gdf_ACLED_21_PA, gdf_ZIP_PA, how="inner", op='within')


In [8]:
#Store Data in GeoJSON file 
output_directory_1 = os.path.join(project_root_dir, '1_Features/b_Conflict_Features/Conflict_PreviousYear')
output_filename_1 = "joined_ACLED_ZIP_21_PA.geojson"
output_path_1 = os.path.join(output_directory_1, output_filename_1)

spatial_join_PA_21.to_file(output_path_1, driver='GeoJSON')

#Transforming Geojson file into GeoDataframe
gdf_PA_21 = gpd.read_file(os.path.join(project_root_dir, '1_Features/b_Conflict_Features/Conflict_PreviousYear/joined_ACLED_ZIP_21_PA.geojson'))

#Converting Geometries to WKT
gdf_PA_21['geometry'] = gdf_PA_21['geometry'].apply(lambda x: x.wkt)

#Saving as csv
output_directory_1 = os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets')
output_filename_1 = "joined_ACLED_ZIP_21_PA.csv"
output_path_1 = os.path.join(output_directory_1, output_filename_1)

gdf_PA_21.to_csv(output_path_1, index=False)

/var/folders/4s/d5gptfss61l9fqz7rk3r5m880000gn/T/ipykernel_33695/957250706.py:12: UserWarning: Geometry column does not contain geometry.
  gdf_PA_21['geometry'] = gdf_PA_21['geometry'].apply(lambda x: x.wkt)


### Restructuring data to extract conflict events per month

In [9]:
#Loading data
ACLED_21_df = pd.read_csv(os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets/joined_ACLED_ZIP_21_PA.csv'))

In [10]:
#Extracting year and month from date column

#Parsing date column to datetime
ACLED_21_df['parsed_date'] = pd.to_datetime(ACLED_21_df['event_date'], format='%d %B %Y')

#Extracting year and month
ACLED_21_df['event_MY'] = ACLED_21_df['parsed_date'].dt.to_period('M')

In [11]:
#Aggregation function for nr of events

#Defining aggregation across features
aggregation_function_21 = {
    'admin2': 'first',  
}

#### Grouping by ZIP Code and Month

In [12]:
#Grouping by ZIP Code and Month
#agg aggregates over the previously specified parameters
grouped_21 = ACLED_21_df.groupby(['ZCTA5CE10', 'event_MY']).agg(aggregation_function_21).reset_index()

#Adding event count per ZIP Code and Month
grouped_21['event_count'] = ACLED_21_df.groupby(['ZCTA5CE10', 'event_MY']).size().reset_index(name='event_count')['event_count']

print(grouped_21.head(10))

   ZCTA5CE10 event_MY     admin2  event_count
0      15009  2021-01     Beaver            1
1      15009  2021-03     Beaver            1
2      15009  2021-05     Beaver            1
3      15014  2021-03  Allegheny            1
4      15014  2021-04  Allegheny            1
5      15017  2021-01  Allegheny            1
6      15017  2021-02  Allegheny            1
7      15017  2021-09  Allegheny            1
8      15024  2021-03  Allegheny            1
9      15044  2021-03  Allegheny            1


In [13]:
#Date range for 2021

date_range_21 = pd.period_range(start='2021-01', end='2021-12', freq='M')
print(date_range_21)

PeriodIndex(['2021-01', '2021-02', '2021-03', '2021-04', '2021-05', '2021-06',
             '2021-07', '2021-08', '2021-09', '2021-10', '2021-11', '2021-12'],
            dtype='period[M]')


In [14]:
#Dataframe mapping ZIP Codes to Counties 
zip_county_21 = ACLED_21_df[['ZCTA5CE10', 'admin2']].drop_duplicates()

#New dataframe combining date range and ZIP Code
all_combinations_21 = pd.MultiIndex.from_product([ACLED_21_df['ZCTA5CE10'].unique(), date_range_21], names=['ZCTA5CE10', 'event_MY'])

#Initializing df with all combinations 
restructured_df_21 = pd.DataFrame(index=all_combinations_21).reset_index()

In [15]:
#Merge 1
#Merging zip_county df with initialized df for all combinations
restructured_df_21 = restructured_df_21.merge(zip_county_21, on='ZCTA5CE10', how='left')

In [16]:
#Merge 2 
#Merging initialized df with grouped data holding aggregates and counts
restructured_df_21 = restructured_df_21.merge(grouped_21, on=['ZCTA5CE10', 'admin2', 'event_MY'], how='left')

In [17]:
#Filling missing values + converting to int
restructured_df_21['event_count'] = restructured_df_21['event_count'].fillna(0).astype(int)

In [18]:
#Sorting df by ZIP and time
restructured_df_21 = restructured_df_21.sort_values(by=['ZCTA5CE10', 'event_MY'])

#Renaming columns
restructured_df_21.rename(columns={'ZCTA5CE10': 'ZCTA'}, inplace=True)

#TO DO: Possibly some restructuring necessary to add missing ZCTAs

In [19]:
#Exporting to csv
output_directory_1 = os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets')
output_filename_1 = "ACLED_Events_21_Clean.csv"
output_path_1 = os.path.join(output_directory_1, output_filename_1)

restructured_df_21.to_csv(output_path_1, index=False)

#### Fill in ZIP Codes that did not have a conflict event in 2021

In [37]:
#Merging dataset with all other ZIPs that did not have conflicts in 2021 

# Load the dataset that was crated in 'all_zips' notebook (12_zip_ACS.csv).
# This holds all ZIP codes, 12 times each, with all ACS data.
final_12_ACS = pd.read_excel(os.path.join(project_root_dir, '2_Dataset/a_Input_Datasets/Template_AllZCTA_Month_21.xlsx'))

conflict_21 = pd.read_csv(os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets/ACLED_Events_21_Clean.csv'))

In [38]:
# Create list of all ZIP codes, taken from the ACS dataset.
zip_list = pd.read_excel(os.path.join(project_root_dir, '2_Dataset/a_Input_Datasets/Template_AllZCTA_Month_21.xlsx'))['ZCTA'].tolist()

In [39]:
# Create a list of the ZIPs that are missing from the conflict set, by literally subtracting the two sets from each other.

missing_zip = list(set(zip_list) - set(conflict_21['ZCTA']))

# Create it into df
missing_zip = pd.DataFrame(missing_zip, columns=['ZCTA'])

# Now, apply same logic as 'all_zips' notebook to create a dataset with all ZIPs, 12 times each. FOR EACH MISSING ZIP
months = pd.date_range(start='2021-01-01', end='2021-12-01', freq='MS').strftime('%Y-%m').tolist()

monthly_missing = pd.DataFrame()

for index, row in missing_zip.iterrows():
    zip_code = row['ZCTA']
    zip_code_entries = pd.DataFrame({'ZCTA': [zip_code] * len(months), 'Month': months})
    monthly_missing = pd.concat([monthly_missing, zip_code_entries])


In [40]:
# Full outer merge the missing (x12) ZIPs with the conflict data.
combined_df = pd.merge(conflict_21, monthly_missing, on='ZCTA', how='outer')

In [41]:
# Fix month columns
combined_df['month'] = combined_df['event_MY'].fillna(combined_df['Month'])
# Drop the old columns
combined_df.drop(['event_MY', 'Month'], axis=1, inplace=True)
# Reorder the variable order
combined_df = combined_df[['ZCTA', 'month', 'admin2', 'event_count']]
# Fill nans from merge to 0 (not present in ACLED, no conflict)
combined_df.fillna(0, inplace=True)

#renaming conflict count column 
combined_df.rename(columns={'event_count': 'event_count_prev_year'}, inplace=True)

In [42]:
#Converting to csv + saving in Folder: b_Partial_Datasets
output_directory_1 = os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets')
output_filename_1 = "ACLED_conflict_previousyear_allZCTA.csv"
output_path_1 = os.path.join(output_directory_1, output_filename_1)

combined_df.to_csv(output_path_1, index=False)